In [1]:
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
!nvidia-smi

Fri Aug 30 00:57:12 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 NVL                On  |   00000000:C6:00.0 Off |                    0 |
| N/A   54C    P0             71W /  400W |       1MiB /  95830MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
model_id = "meta-llama/Meta-Llama-3.1-8B"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

# Set the pad_token_id to eos_token_id
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    device="cuda"
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
# Tokenizer https://github.com/meta-llama/llama/blob/main/llama/tokenizer.py
import os
from typing import List

from sentencepiece import SentencePieceProcessor

class Tokenizer:
    def __init__(self, model_path: str):
        assert os.path.isfile(model_path), model_path
        self.sp_model = SentencePieceProcessor(model_file=model_path)

        self.n_words: int = self.sp_model.vocab_size()
        self.bos_id: int = self.sp_model.bos_id()
        self.eos_id: int = self.sp_model.eos_id()
        self.pad_id: int = self.sp_model.pad_id()

        assert self.sp_model.vocab_size() == self.sp_model.get_piece_size()

    def encode(self, s: str, bos: bool, eos: bool) -> List[int]:
        assert type(s) is str
        t = self.sp_model.encode(s)
        if bos:
            t = [self.bos_id] + t
        if eos:
            t = t + [self.eos_id]
        return t

    def decode(self, t: List[int]) -> str:
        return self.sp_model.decode(t)


In [5]:
import time
import random
from transformers import PreTrainedTokenizerBase

def generate_random_tokens(tokenizer: PreTrainedTokenizerBase, num_tokens: int) -> List[int]:
    # Get the vocabulary size
    vocab_size = len(tokenizer.get_vocab())

    # Generate random token IDs
    random_tokens = [random.randint(0, vocab_size - 1) for _ in range(num_tokens)]

    return random_tokens    
# List of inputs with varying lengths
inputs = [
    generate_random_tokens(tokenizer=tokenizer, num_tokens=10), 
    generate_random_tokens(tokenizer=tokenizer,num_tokens=100), 
    generate_random_tokens(tokenizer=tokenizer,num_tokens=2000),  
    generate_random_tokens(tokenizer=tokenizer,num_tokens=5000),  
    generate_random_tokens(tokenizer=tokenizer,num_tokens=10000),  
    generate_random_tokens(tokenizer=tokenizer,num_tokens=20000),  
]
print(inputs[0])

[15808, 53778, 70053, 19177, 104839, 94388, 108523, 123723, 123178, 54660]


In [6]:
def run_model(input_tokens, min_length, max_length):
    start_time = time.time()
    
    # Convert input tokens to tensor
    input_ids = torch.tensor([input_tokens]).to(model.device)
    
    # Create attention mask
    attention_mask = torch.ones_like(input_ids)
    
    # Generate
    with torch.no_grad():
        output = model.generate(
            input_ids,
            attention_mask=attention_mask,
            min_length= min_length,
            max_length=max_length,
            do_sample=True,
        )
    
    end_time = time.time()
    return output, end_time - start_time

# Run the benchmark
for input_tokens in inputs:
    total_time = 0
    print(f"\nInput length: {len(input_tokens)}")
    for i in range(5):
        output, duration = run_model(input_tokens, min_length=len(input_tokens)+100, max_length=len(input_tokens)+100)
        total_time += duration
        print(f"  Run {i+1}: Time: {duration:.4f} seconds, New Output length: {output.shape[1]-len(input_tokens)}")
    
    average_time = total_time / 5
    print(f"Average time: {average_time:.4f} seconds")


Input length: 10
  Run 1: Time: 2.0059 seconds, New Output length: 100
  Run 2: Time: 1.7029 seconds, New Output length: 100
  Run 3: Time: 1.7066 seconds, New Output length: 100
  Run 4: Time: 1.7049 seconds, New Output length: 100
  Run 5: Time: 1.6980 seconds, New Output length: 100
Average time: 1.7637 seconds

Input length: 100
  Run 1: Time: 1.7337 seconds, New Output length: 100
  Run 2: Time: 1.7341 seconds, New Output length: 100
  Run 3: Time: 1.7333 seconds, New Output length: 100
  Run 4: Time: 1.7246 seconds, New Output length: 100
  Run 5: Time: 1.7238 seconds, New Output length: 100
Average time: 1.7299 seconds

Input length: 2000
  Run 1: Time: 3.1746 seconds, New Output length: 100
  Run 2: Time: 3.1766 seconds, New Output length: 100
  Run 3: Time: 3.1728 seconds, New Output length: 100
  Run 4: Time: 3.1714 seconds, New Output length: 100
  Run 5: Time: 3.1714 seconds, New Output length: 100
Average time: 3.1734 seconds

Input length: 5000
  Run 1: Time: 5.9145 seco

In [7]:
def run_model(input_tokens, min_length, max_length):
    start_time = time.time()
    
    # Convert input tokens to tensor
    input_ids = torch.tensor([input_tokens]).to(model.device)
    
    # Create attention mask
    attention_mask = torch.ones_like(input_ids)
    
    # Generate
    with torch.no_grad():
        output = model.generate(
            input_ids,
            attention_mask=attention_mask,
            min_length= min_length,
            max_length=max_length,
            do_sample=True,
        )
    
    end_time = time.time()
    return output, end_time - start_time

# Run the benchmark
for input_tokens in inputs:
    total_time = 0
    print(f"\nInput length: {len(input_tokens)}")
    for i in range(5):
        output, duration = run_model(input_tokens, min_length=len(input_tokens)+1000, max_length=len(input_tokens)+1000)
        total_time += duration
        print(f"  Run {i+1}: Time: {duration:.4f} seconds, New Output length: {output.shape[1]-len(input_tokens)}")
    
    average_time = total_time / 5
    print(f"Average time: {average_time:.4f} seconds")


Input length: 10
  Run 1: Time: 17.5054 seconds, New Output length: 1000
  Run 2: Time: 17.4915 seconds, New Output length: 1000
  Run 3: Time: 17.4752 seconds, New Output length: 1000
  Run 4: Time: 17.4888 seconds, New Output length: 1000
  Run 5: Time: 17.4860 seconds, New Output length: 1000
Average time: 17.4894 seconds

Input length: 100
  Run 1: Time: 17.7314 seconds, New Output length: 1000
  Run 2: Time: 17.7357 seconds, New Output length: 1000
  Run 3: Time: 17.7968 seconds, New Output length: 1000
  Run 4: Time: 17.7309 seconds, New Output length: 1000
  Run 5: Time: 17.8956 seconds, New Output length: 1000
Average time: 17.7781 seconds

Input length: 2000
  Run 1: Time: 26.9129 seconds, New Output length: 1000
  Run 2: Time: 26.9237 seconds, New Output length: 1000
  Run 3: Time: 26.8962 seconds, New Output length: 1000
  Run 4: Time: 26.8759 seconds, New Output length: 1000
  Run 5: Time: 26.8888 seconds, New Output length: 1000
Average time: 26.8995 seconds

Input length

KeyboardInterrupt: 